# Phase 1 Comparison of Burn Severity Products: Non-QA vs With-QA

## 1. Objective

This notebook compares **Non-QA** and **With-QA** burn severity products in terms of:

* Total burned area
* Distribution of burn severity levels
* Top-3 affected land-cover types per severity level

The goal is to assess whether QA filtering alters **management-relevant conclusions**.

---

## 2. Inputs

* **Burn severity polygons (Non-QA)**
* **Burn severity polygons (With-QA)**
* **Natural land use / land cover dataset**

## 3. Load libraries

In [ ]:
import os
import pandas as pd

## 4. Paths

In [9]:
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
OUTPUT_DIR = os.path.join(BASE_DIR, 'output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

NONQA_DIR = os.path.join(OUTPUT_DIR, 'non_qa')
WITHQA_DIR = os.path.join(OUTPUT_DIR, 'with_qa')

AREA_NONQA_PATH = os.path.join(NONQA_DIR, 'burn_severity_area_summary_non_qa.csv')
TOP3_NONQA_PATH = os.path.join(NONQA_DIR, "top_3_landuse_non_qa.csv")

AREAQA_PATH = os.path.join(WITHQA_DIR, 'burn_severity_area_summary_qa.csv')
TOP3QA_PATH = os.path.join(WITHQA_DIR, "top_3_landuse_qa.csv")

## 5. Load summary tables

In [ ]:
# Non-QA summary
nonqa = pd.read_csv(AREA_NONQA_PATH)

# With-QA summary
withqa = pd.read_csv(AREAQA_PATH)

## 6. Total burned area per severity level

In [22]:
comparison_totals = nonqa.merge(
    withqa,
    on="severity_label",
    suffixes=("_nonqa", "_withqa")
).drop(['severity_nonqa', 'severity_withqa'], axis=1)


In [23]:
total_nonqa = comparison_totals["area_ha_nonqa"].sum()
total_withqa = comparison_totals["area_ha_withqa"].sum()

print(total_nonqa, total_withqa)

74137.28 74137.28


In [26]:
comparison_totals["difference_ha"] = (
    comparison_totals["area_ha_nonqa"] - comparison_totals["area_ha_withqa"]
)

comparison_totals["perc_difference"] = (
    (comparison_totals["difference_ha"] / comparison_totals["area_ha_nonqa"]) * 100
)

print(comparison_totals)

  severity_label  area_ha_nonqa  area_ha_withqa  difference_ha  \
0       Unburned       44791.04        37686.64        7104.40   
1            Low        4533.48         5713.12       -1179.64   
2   Moderate-Low        4828.40        15335.92      -10507.52   
3  Moderate-High        6477.92        14672.48       -8194.56   
4           High       13506.44          729.12       12777.32   

   perc_difference  
0        15.861208  
1       -26.020629  
2      -217.619087  
3      -126.499864  
4        94.601686  


## 7. Load top-3 land use

In [ ]:
# Non-QA top-3 landuse
nonqa_top3 = pd.read_csv(TOP3_NONQA_PATH)

# With-QA top-3 landuse
withqa_top3 = pd.read_csv(TOP3QA_PATH)

## 8. Compare dominant land-cover patterns

In [ ]:
comparison_top3 = nonqa_top3.merge(
    withqa_top3,
    on=["severity_label", "info"],
    how="outer",
    suffixes=("_nonqa", "_withqa")
).fillna(0)

comparison_top3["difference_ha"] = (
    comparison_top3["area_ha_nonqa"] - comparison_top3["area_ha_withqa"]
)

print(comparison_top3)

,severity_label,info,area_ha_nonqa,area_ha_withqa,difference_ha
0,High,311-Broad-leaved forest,1763.930401,179.237972,1584.692429
1,High,322-Moors and heathland,6808.751794,295.698969,6513.052825
2,High,324-Transitional woodland shrub,1805.605507,133.955388,1671.650119
3,Low,311-Broad-leaved forest,0.000000,263.197738,-263.197738
4,Low,312-Coniferous forest,292.172702,392.162711,-99.990009
5,Low,322-Moors and heathland,2465.189882,1571.535929,893.653953
6,Low,324-Transitional woodland shrub,247.426189,0.000000,247.426189
7,Moderate-High,311-Broad-leaved forest,573.646356,0.000000,573.646356
8,Moderate-High,312-Coniferous forest,0.000000,1531.980118,-1531.980118
9,Moderate-High,322-Moors and heathland,3849.594779,8251.675413,-4402.080634


## 8. Interpretation

### Key observations:

- QA filtering substantially **reduces mapped High burn severity** (approximately **–95%**), with affected areas predominantly **redistributed into Moderate severity classes and Unburned**. Low burn severity shows a slight increase after QA filtering (+1,180 ha), indicating reclassification of marginal burn signals.

- QA filtering **alters both the extent and composition** of burn severity classes. While High-severity impacts remain concentrated in the same land-cover types (moors and heathland, transitional woodland–shrub, and forest), large portions of **shrubland and forest are reassigned to Moderate-High and Moderate-Low** severity classes, resulting in changes to the top-3 dominant land-cover types at moderate severity levels.

- As a result, non-QA products likely overestimate extreme fire damage, while QA-filtered products provide a more conservative and reliable basis for damage assessment and recovery planning.

### Management implication:

- With-QA data should be used for damage estimation, recovery prioritization, and compensation or reporting, as severity-dependent impacts are highly sensitive to QA filtering.

- Land-cover–based mitigation and prevention priorities (e.g. focus on moors, heathland, shrubland, and forest–shrub transition zones) remain robust across QA and non-QA products, supporting their use for strategic fire-risk management.



## 9. Conclusion

> QA filtering substantially alters burn-severity attribution.

> While dominant land-cover patterns are broadly consistent, QA filtering significantly changes burn-severity classification, making QA-products more suitable for reliable, severity-based decision making.

